In [ ]:
# ============================================================
# Incremental Data Analysis & Insight Generation System
# Advanced Google Colab Version
# ============================================================

# Install libraries if needed
!pip install pandas numpy matplotlib scikit-learn scipy openpyxl -q

# ============================================================
# IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from scipy.stats import ttest_ind

from google.colab import files
from IPython.display import display

import warnings
warnings.filterwarnings("ignore")

# ============================================================
# FILE UPLOAD SECTION
# ============================================================

print("========================================")
print("UPLOAD HISTORICAL DATASET FILE")
print("========================================")

uploaded_old = files.upload()

old_file_name = list(uploaded_old.keys())[0]

print("\nUploaded File:", old_file_name)

print("\n========================================")
print("UPLOAD NEW DATASET FILE")
print("========================================")

uploaded_new = files.upload()

new_file_name = list(uploaded_new.keys())[0]

print("\nUploaded File:", new_file_name)

# ============================================================
# MAIN SYSTEM CLASS
# ============================================================

class IncrementalInsightSystem:

    def __init__(self):

        self.old_data = None
        self.new_data = None

        self.scaler = MinMaxScaler()

    # ========================================================
    # SMART FILE READER
    # ========================================================

    def smart_read(self, file_path):

        if file_path.endswith('.xlsx') or file_path.endswith('.xls'):

            return pd.read_excel(file_path)

        else:

            try:

                return pd.read_csv(file_path)

            except UnicodeDecodeError:

                return pd.read_csv(
                    file_path,
                    encoding='latin1'
                )

    # ========================================================
    # LOAD DATASETS
    # ========================================================

    def load_datasets(self, old_path, new_path):

        print("\n========================================")
        print("LOADING DATASETS")
        print("========================================")

        self.old_data = self.smart_read(old_path)

        self.new_data = self.smart_read(new_path)

        # Dataset validation
        if self.old_data.empty or self.new_data.empty:

            print("\nERROR: One of the datasets is empty.")

            return

        print("\nOld Dataset Shape :", self.old_data.shape)

        print("New Dataset Shape :", self.new_data.shape)

        print("\nOld Dataset Preview:")
        display(self.old_data.head())

        print("\nNew Dataset Preview:")
        display(self.new_data.head())

    # ========================================================
    # PREPROCESSING
    # ========================================================

    def preprocess(self):

        print("\n========================================")
        print("PREPROCESSING STARTED")
        print("========================================")

        # ---------------------------
        # DUPLICATE REMOVAL
        # ---------------------------

        old_duplicates = self.old_data.duplicated().sum()

        new_duplicates = self.new_data.duplicated().sum()

        self.old_data.drop_duplicates(inplace=True)

        self.new_data.drop_duplicates(inplace=True)

        print(f"\nRemoved {old_duplicates} duplicates from OLD dataset")

        print(f"Removed {new_duplicates} duplicates from NEW dataset")

        # ---------------------------
        # MISSING VALUE HANDLING
        # ---------------------------

        numeric_old = self.old_data.select_dtypes(
            include=np.number
        ).columns

        numeric_new = self.new_data.select_dtypes(
            include=np.number
        ).columns

        for col in numeric_old:

            self.old_data[col].fillna(
                self.old_data[col].mean(),
                inplace=True
            )

        for col in numeric_new:

            self.new_data[col].fillna(
                self.new_data[col].mean(),
                inplace=True
            )

        print("\nMissing values handled successfully")

        # ---------------------------
        # COMMON NUMERIC COLUMNS
        # ---------------------------

        self.common_numeric = list(

            set(numeric_old).intersection(

                set(numeric_new)

            )

        )

        print("\nCommon Numeric Columns Found:")

        print(self.common_numeric)

        # ---------------------------
        # NORMALIZATION
        # ---------------------------

        self.old_data[self.common_numeric] = self.scaler.fit_transform(

            self.old_data[self.common_numeric]

        )

        self.new_data[self.common_numeric] = self.scaler.transform(

            self.new_data[self.common_numeric]

        )

        print("\nNormalization completed")

    # ========================================================
    # STATISTICAL ANALYSIS
    # ========================================================

    def compare_statistics(self):

        print("\n========================================")
        print("STATISTICAL COMPARISON")
        print("========================================")

        results = []

        for col in self.common_numeric:

            old_mean = self.old_data[col].mean()

            new_mean = self.new_data[col].mean()

            old_std = self.old_data[col].std()

            new_std = self.new_data[col].std()

            percent_change = (

                ((new_mean - old_mean) / old_mean) * 100

                if old_mean != 0 else 0

            )

            t_stat, p_value = ttest_ind(

                self.old_data[col],

                self.new_data[col],

                equal_var=False

            )

            significant = (

                "Yes"

                if p_value < 0.05

                else "No"

            )

            results.append({

                "Feature": col,

                "Old Mean": round(old_mean, 4),

                "New Mean": round(new_mean, 4),

                "Old Std": round(old_std, 4),

                "New Std": round(new_std, 4),

                "% Change": round(percent_change, 2),

                "P-Value": round(p_value, 5),

                "Statistically Significant": significant
            })

        result_df = pd.DataFrame(results)

        print("\nStatistical Comparison Results:\n")

        display(result_df)

        # Export results
        result_df.to_csv(

            "analysis_results.csv",

            index=False

        )

        print("\nAnalysis results exported successfully")

        return result_df

    # ========================================================
    # TREND ANALYSIS
    # ========================================================

    def trend_analysis(self):

        print("\n========================================")
        print("TREND ANALYSIS")
        print("========================================")

        trend_results = []

        for col in self.common_numeric:

            old_mean = self.old_data[col].mean()

            new_mean = self.new_data[col].mean()

            if new_mean > old_mean:

                trend = "Increasing"

            elif new_mean < old_mean:

                trend = "Decreasing"

            else:

                trend = "Stable"

            trend_results.append({

                "Feature": col,

                "Trend": trend
            })

        trend_df = pd.DataFrame(trend_results)

        display(trend_df)

    # ========================================================
    # VISUAL ANALYTICS
    # ========================================================

    def visualize_comparison(self):

        print("\n========================================")
        print("VISUAL ANALYTICS")
        print("========================================")

        selected_columns = self.common_numeric[:4]

        for col in selected_columns:

            plt.figure(figsize=(10, 5))

            plt.plot(

                self.old_data[col].values,

                label="Historical Data"

            )

            plt.plot(

                self.new_data[col].values,

                label="New Data"

            )

            plt.title(f"Trend Comparison - {col}")

            plt.xlabel("Records")

            plt.ylabel("Normalized Values")

            plt.legend()

            plt.grid(True)

            plt.show()

    # ========================================================
    # CORRELATION ANALYSIS
    # ========================================================

    def correlation_analysis(self):

        print("\n========================================")
        print("CORRELATION ANALYSIS")
        print("========================================")

        corr = self.new_data[
            self.common_numeric
        ].corr()

        plt.figure(figsize=(12, 8))

        plt.imshow(

            corr,

            cmap="coolwarm",

            aspect='auto'

        )

        plt.colorbar()

        plt.xticks(

            range(len(corr.columns)),

            corr.columns,

            rotation=90

        )

        plt.yticks(

            range(len(corr.columns)),

            corr.columns

        )

        plt.title("Correlation Heatmap")

        plt.show()

    # ========================================================
    # ANOMALY DETECTION
    # ========================================================

    def anomaly_detection(self):

        print("\n========================================")
        print("ANOMALY DETECTION")
        print("========================================")

        anomaly_results = []

        for col in self.common_numeric:

            mean = self.new_data[col].mean()

            std = self.new_data[col].std()

            upper_limit = mean + 3 * std

            lower_limit = mean - 3 * std

            anomalies = self.new_data[

                (self.new_data[col] > upper_limit)

                |

                (self.new_data[col] < lower_limit)

            ]

            anomaly_results.append({

                "Feature": col,

                "Anomaly Count": len(anomalies)
            })

        anomaly_df = pd.DataFrame(anomaly_results)

        display(anomaly_df)

    # ========================================================
    # FINAL INSIGHTS
    # ========================================================

    def generate_insights(self):

        print("\n========================================")
        print("FINAL INSIGHTS")
        print("========================================")

        for col in self.common_numeric:

            old_mean = self.old_data[col].mean()

            new_mean = self.new_data[col].mean()

            if new_mean > old_mean:

                print(f"\n{col} shows growth in new dataset")

            elif new_mean < old_mean:

                print(f"\n{col} shows decline in new dataset")

            else:

                print(f"\n{col} remains stable")

    # ========================================================
    # DOWNLOAD RESULTS
    # ========================================================

    def download_results(self):

        print("\n========================================")
        print("DOWNLOADING RESULTS")
        print("========================================")

        files.download("analysis_results.csv")

# ============================================================
# SYSTEM EXECUTION
# ============================================================

system = IncrementalInsightSystem()

system.load_datasets(

    old_file_name,

    new_file_name

)

system.preprocess()

system.compare_statistics()

system.trend_analysis()

system.visualize_comparison()

system.correlation_analysis()

system.anomaly_detection()

system.generate_insights()

system.download_results()

print("\n========================================")
print("SYSTEM EXECUTION COMPLETED")
print("========================================")